# 00 — Human Detection & Cropping Module (YOLOv8n)

Modul persiapan data untuk `identity-lora-pipeline`.

Alur kerja:
1. **Deteksi manusia** pada gambar mentah menggunakan model **YOLOv8n** (kelas `person`).
2. Setiap deteksi menghasilkan **bounding box** `(x1, y1, x2, y2)` beserta skor kepercayaan.
3. Koordinat bounding box tersebut dikirim ke **modul cropping**, yang memotong region gambar sesuai bounding box (dengan padding opsional) dan menyimpannya sebagai gambar identitas individu.

Sebagai contoh uji coba pipeline, kita memakai dua identitas contoh: **red-chan** dan **pink-chan**. Ganti `IDENTITIES` di bawah dengan folder gambar identitas nyata pada dataset kalian.

## 1. Instalasi & Import Dependensi

In [ ]:
%pip install -q ultralytics opencv-python-headless numpy

In [ ]:
import os
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
from ultralytics import YOLO

## 2. Konfigurasi Path

In [ ]:
RAW_DATA_DIR = Path("data/raw")
CROPPED_DATA_DIR = Path("data/cropped")
MODEL_WEIGHTS = "yolov8n.pt"

# Identitas contoh untuk menguji pipeline deteksi + cropping.
# Setiap key adalah nama identitas, value adalah folder berisi gambar mentah identitas tsb.
IDENTITIES = {
    "red-chan": RAW_DATA_DIR / "red-chan",
    "pink-chan": RAW_DATA_DIR / "pink-chan",
}

CROPPED_DATA_DIR.mkdir(parents=True, exist_ok=True)

## 3. Modul Deteksi Manusia (YOLOv8n)

Kelas `HumanDetector` membungkus model YOLOv8n dan mengembalikan daftar bounding box untuk setiap orang yang terdeteksi pada sebuah gambar.

In [ ]:
COCO_PERSON_CLASS_ID = 0


@dataclass
class BoundingBox:
    x1: int
    y1: int
    x2: int
    y2: int
    confidence: float


class HumanDetector:
    def __init__(self, weights: str = MODEL_WEIGHTS, conf_threshold: float = 0.5):
        self.model = YOLO(weights)
        self.conf_threshold = conf_threshold

    def detect(self, image_path: Path) -> list[BoundingBox]:
        results = self.model.predict(
            source=str(image_path),
            classes=[COCO_PERSON_CLASS_ID],
            conf=self.conf_threshold,
            verbose=False,
        )

        boxes: list[BoundingBox] = []
        for result in results:
            for box in result.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                boxes.append(
                    BoundingBox(
                        x1=int(x1),
                        y1=int(y1),
                        x2=int(x2),
                        y2=int(y2),
                        confidence=float(box.conf[0]),
                    )
                )
        return boxes

## 4. Modul Cropping

Menerima koordinat bounding box dari modul deteksi dan memotong region tersebut dari gambar asli, dengan padding opsional agar hasil crop tidak terlalu ketat pada tubuh.

In [ ]:
class Cropper:
    def __init__(self, padding_ratio: float = 0.1):
        self.padding_ratio = padding_ratio

    def crop(self, image: np.ndarray, bbox: BoundingBox) -> np.ndarray:
        h, w = image.shape[:2]
        box_w = bbox.x2 - bbox.x1
        box_h = bbox.y2 - bbox.y1
        pad_x = int(box_w * self.padding_ratio)
        pad_y = int(box_h * self.padding_ratio)

        x1 = max(bbox.x1 - pad_x, 0)
        y1 = max(bbox.y1 - pad_y, 0)
        x2 = min(bbox.x2 + pad_x, w)
        y2 = min(bbox.y2 + pad_y, h)

        return image[y1:y2, x1:x2]

## 5. Pipeline: Deteksi -> Cropping

Menggabungkan `HumanDetector` dan `Cropper`: bounding box hasil deteksi langsung dikirim ke modul cropping untuk setiap gambar pada folder identitas.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}


def process_identity(identity_name: str, source_dir: Path, detector: HumanDetector, cropper: Cropper) -> int:
    output_dir = CROPPED_DATA_DIR / identity_name
    output_dir.mkdir(parents=True, exist_ok=True)

    if not source_dir.exists():
        print(f"[skip] folder tidak ditemukan untuk '{identity_name}': {source_dir}")
        return 0

    crop_count = 0
    for image_path in sorted(source_dir.iterdir()):
        if image_path.suffix.lower() not in IMAGE_EXTENSIONS:
            continue

        image = cv2.imread(str(image_path))
        if image is None:
            print(f"[warn] gagal membaca gambar: {image_path}")
            continue

        boxes = detector.detect(image_path)
        for idx, bbox in enumerate(boxes):
            cropped = cropper.crop(image, bbox)
            if cropped.size == 0:
                continue

            out_path = output_dir / f"{image_path.stem}_person{idx}.jpg"
            cv2.imwrite(str(out_path), cropped)
            crop_count += 1

    print(f"[{identity_name}] {crop_count} crop tersimpan di {output_dir}")
    return crop_count

## 6. Jalankan Pipeline untuk Identitas Contoh (red-chan, pink-chan)

In [ ]:
detector = HumanDetector(weights=MODEL_WEIGHTS, conf_threshold=0.5)
cropper = Cropper(padding_ratio=0.1)

total_crops = 0
for identity_name, source_dir in IDENTITIES.items():
    total_crops += process_identity(identity_name, source_dir, detector, cropper)

print(f"Total crop dihasilkan: {total_crops}")

## 7. Catatan

- Letakkan gambar mentah tiap identitas di `data/raw/<nama-identitas>/`.
- Hasil crop akan tersimpan otomatis di `data/cropped/<nama-identitas>/`.
- Ganti `MODEL_WEIGHTS` dengan path checkpoint YOLOv8n custom jika sudah difine-tune.
- Notebook berikutnya pada pipeline (mis. `01_face_alignment.ipynb`) dapat langsung memakai hasil crop di `data/cropped/` sebagai input.